# Neural Networks in scikit-learn

This notebook focuses on instead of the working of a nn in itself:

1. Understanding neural network components in scikit-learn
2. Parameters vs fitted attributes
3. Optimization methods (Adam, SGD, LBFGS)
4. Practical usage and interpretation

Goal:
Understand how neural networks are implemented and controlled in practice.

In [ ]:
# DATA PREPARATION, yes we need to do this all the time

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

data = load_iris()

X = data.data
y = data.target

# Important: scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:

from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(50,),
    max_iter=500
)

model.fit(X_train, y_train)


## Parameters vs Fitted Attributes

### Parameters (you set them)

Defined before training:

- hidden_layer_sizes --> architecture
- activation --> activation function
- solver --> optimization method
- learning_rate_init --> step size
- max_iter --> training iterations

These control how the model learns.

### Fitted Attributes (learned by the model)

After training:

- coefs_ --> weight matrices
- intercepts_ --> biases
- loss_curve_ --> error over time

These represent what the model learned from data.


### Key Insight

Parameters = instructions  
Fitted attributes = learned knowledge


In [ ]:
# INSPECT FITTED MODEL

print("Number of layers:", len(model.coefs_))
print("Weights shape:", [w.shape for w in model.coefs_])

print("First layer weights:")
print(model.coefs_[0])

In [ ]:
#Loss Curve
import matplotlib.pyplot as plt

plt.plot(model.loss_curve_)
plt.title("Training Loss Curve")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()

## Optimization Methods in Neural Networks

 1. Adam (default)

- adaptive learning rates
- combines momentum + scaling

 stable  
 works well in most cases, we love adam  


### 2. SGD (Stochastic Gradient Descent)

- simple gradient updates
- requires manual tuning

flexible  
sensitive to learning rate  


### 3. LBFGS

- second-order optimization
- no mini-batches

works well for small datasets  
not scalable  


### Key Insight

Different optimizers = different ways of updating weights

In [ ]:
# COMPARING OPTIMIZERS

solvers = ["adam", "sgd", "lbfgs"]

for solver in solvers:
    
    model = MLPClassifier(
        hidden_layer_sizes=(50,),
        solver=solver,
        max_iter=500,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    print(f"{solver}: final loss = {model.loss_:.4f}")


## Learning Rate

Controls how big each update step is:

- too small --> slow learning  
- too large --> unstable training  


### Analogy

Walking downhill:

- small steps --> slow  
- large steps --> might fall  


### Key Insight

Learning rate strongly affects convergence behavior

In [ ]:
# LEARNING RATE EXPERIMENT

rates = [0.001, 0.01, 0.1]

for lr in rates:
    
    model = MLPClassifier(
        solver="sgd",
        learning_rate_init=lr,
        hidden_layer_sizes=(50,),
        max_iter=500,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    print(f"LR {lr}: final loss = {model.loss_:.4f}")

## Evaluation Metrics for Classification


### 1. Why Accuracy is Not Enough

Accuracy measures:

    correct predictions / total predictions

Problem:
- ignores class imbalance  
- does not show types of errors  


### 2. Precision

Measures:

    When the model predicts a class, how often is it correct?

High precision = few false positives


### 3. Recall

Measures:

    How many true samples did we correctly find?

High recall = few false negatives


### 4. F1 Score

Balances precision and recall:

    F1 = harmonic mean of precision and recall

Useful when both types of errors matter.


### 5. Log Loss (Cross-Entropy)

Measures quality of predicted probabilities:

- correct + confident → low loss  
- wrong + confident → very high loss  


### 6. Confusion Matrix

Shows:

    actual vs predicted values

Helps identify:

- which classes are confused  
- types of errors  


### Key Insight

Evaluation metrics answer:

    How good is this model for my specific problem?

Different metrics highlight different weaknesses.

In [ ]:
# MODEL EVALUATION (aka. we learn about more metrics than accuracy now)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    confusion_matrix,
    classification_report
)

# Predictions
preds = model.predict(X_test)

# Some models also give probabilities
if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)
else:
    probs = None

# BASIC METRICS

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, average="weighted")
recall = recall_score(y_test, preds, average="weighted")
f1 = f1_score(y_test, preds, average="weighted")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# PROBABILITY-BASED METRIC

if probs is not None:
    loss = log_loss(y_test, probs)
    print("Log Loss:", loss)

# CONFUSION MATRIX

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

# FULL CLASSIFICATION REPORT

print("\nClassification Report:")
print(classification_report(y_test, preds))

## Final Understanding

You now understand:

### Neural Networks in scikit-learn consist of:

- Parameters (architecture, optimization choices)
- Optimization process (gradient-based)
- Learned weights (coefs_)


### Training process:

1. Initialize weights  
2. Predict  
3. Compute loss  
4. Update weights  

### Evaluation process
- there are multiple metrics we can use to evaluate to analyse our models perfomance
- each of them is used for different use cases

### Key Concept

Neural networks are NOT magic:

They are:

    parameterized functions optimized using gradients


## ROC Curve and Precision-Recall Curve

### 1. Why We Need These

Accuracy alone is not sufficient.

We need to understand:

- how the model behaves under different thresholds  
- how it balances false positives and false negatives  


## ROC Curve (Receiver Operating Characteristic)

### Idea

Plot:

    True Positive Rate vs False Positive Rate


### Interpretation

- good model --> curve close to top-left  
- random model --> diagonal line  


### AUC (Area Under Curve)

Measures overall performance:

- 1.0 --> perfect  
- 0.5 --> random  

### Intuition

ROC answers:

    How well can the model separate classes?


## Precision-Recall Curve

### Idea

Plot:

    Precision vs Recall


### Interpretation

- shows tradeoff:
    high recall --> more positives found  
    high precision --> fewer false positives  


### When to Use

Better than ROC when:

- classes are imbalanced  
- false positives matter  


## Key Difference

ROC:
    focuses on ranking performance

PR:
    focuses on prediction quality for positive class

## Important Insight

Both curves depend on:

    probability outputs

This is why:

    predict_proba() is critical

In [ ]:
# ROC CURVE + PRECISION-RECALL CURVE

import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import label_binarize

# STEP 1  GET PROBABILITIES again

# Neural networks give probabilities 
y_probs = model.predict_proba(X_test)

# STEP 2  BINARIZE TARGET (multiclass → one-vs-rest)

# Iris has 3 classes --> we convert to binary for each class
classes = np.unique(y)
y_test_bin = label_binarize(y_test, classes=classes)

n_classes = y_test_bin.shape[1]

# ROC CURVE

plt.figure()

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f"Class {i} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], "k--")  # random baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend()
plt.show()

# PRECISION-RECALL CURVE

plt.figure()

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(
        y_test_bin[:, i], y_probs[:, i]
    )
    
    plt.plot(recall, precision, label=f"Class {i}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()
